# 08 Text Representation and Local Qiskit

This post-hackathon portfolio study expands benchmark v2. First reproduce a small **synthetic demonstration** of text, reduction and local circuits; then inspect the full dataset report. Synthetic metrics are not phishing-validation evidence. No provider connection or credentials are used.

In [1]:
import json
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.evaluation.benchmark_v2 import core
from src.evaluation.benchmark_v2_text import features, quantum
from src.features.host_features import host_features

ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
print("Synthetic demonstration: 24 train, 12 calibration, 12 test rows")

Synthetic demonstration: 24 train, 12 calibration, 12 test rows


## Training-only representation

Combine character TF-IDF with normalized numeric features. Learn four SVD components and scaling on training rows only. The generated labels below exist solely to demonstrate the computation.

In [2]:
hosts = [f"{'verify-access' if i % 2 else 'example-shop'}{i}.com" for i in range(48)]
rows = pd.DataFrame({"host": hosts, "is_phishing": np.arange(48) % 2})
rows = pd.concat([rows, pd.DataFrame([host_features(h) for h in hosts])], axis=1)
train, calibration, test = rows.iloc[:24], rows.iloc[24:36], rows.iloc[36:]
rep = features.Representation(seed=42, max_features=500).fit(train)
a, _ = rep.transform(train)
b, _ = rep.transform(calibration)
c, clipping = rep.transform(test)
y_train = train.is_phishing.to_numpy()
y_cal = calibration.is_phishing.to_numpy()
y_test = test.is_phishing.to_numpy()
print(json.dumps(rep.diagnostics(), indent=2))
print("Test clipping:", clipping)

{
  "text_features": 312,
  "explained_variance_ratio": [
    0.3957006878345421,
    0.2683117062719105,
    0.0019484952769319374,
    0.044782113693801445
  ],
  "component_minima": [
    -0.6533389592726087,
    -0.5013764822514715,
    0.2850478857071713,
    -0.7209099765287181
  ],
  "component_maxima": [
    0.6652951407702585,
    0.5933968068091566,
    0.431444430761802,
    0.13817603316600902
  ]
}
Test clipping: {'per_feature': [0, 5, 0, 0], 'rows_with_any_clipped': 5, 'constant_features': [], 'constant_feature_shift_rows': 0}


In [3]:
classical = features.fit_model(
    features.estimator("reduced_rbf", {"C": 1, "gamma": "scale"}, 42),
    a["reduced"],
    y_train,
)
classical_threshold = core.operating_point(y_cal, features.scores(classical, b["reduced"]), 0.1)[
    "threshold"
]
classical_scores = features.scores(classical, c["reduced"])
print("Synthetic classical metrics:")
print(
    json.dumps(
        core.metrics(y_test, core.predict(classical_scores, classical_threshold), classical_scores),
        indent=2,
    )
)

Synthetic classical metrics:
{
  "accuracy": 0.6666666666666666,
  "balanced_accuracy": 0.6666666666666666,
  "precision": 1.0,
  "recall": 0.3333333333333333,
  "false_positive_rate": 0.0,
  "f1": 0.5,
  "average_precision": 1.0,
  "confusion_matrix": [
    [
      6,
      0
    ],
    [
      4,
      2
    ]
  ],
  "rows": 12
}


## Actual Qiskit Statevector

Encode the four components with Z and angle maximum ?/4. The quantum routine constructs and binds Qiskit circuits; a conventional SVM learns from their exact fidelity kernel.

In [4]:
angle_max = np.pi / 4
reference = quantum.states(a["reduced"] * angle_max, "z", "synthetic-demo")
cal_states = quantum.states(b["reduced"] * angle_max, "z", "synthetic-demo")
test_states = quantum.states(c["reduced"] * angle_max, "z", "synthetic-demo")
classifier = features.fit_model(
    features.estimator("quantum", {"C": 1}, 42),
    core.kernel(reference, reference),
    y_train,
)
threshold = core.operating_point(
    y_cal, features.scores(classifier, core.kernel(cal_states, reference)), 0.1
)["threshold"]
ideal_kernel = core.kernel(test_states, reference)
ideal_scores = features.scores(classifier, ideal_kernel)
ideal_predictions = core.predict(ideal_scores, threshold)
print("Synthetic ideal-quantum metrics:")
print(json.dumps(core.metrics(y_test, ideal_predictions, ideal_scores), indent=2))

Synthetic ideal-quantum metrics:
{
  "accuracy": 0.6666666666666666,
  "balanced_accuracy": 0.6666666666666666,
  "precision": 1.0,
  "recall": 0.3333333333333333,
  "false_positive_rate": 0.0,
  "f1": 0.5,
  "average_precision": 1.0,
  "confusion_matrix": [
    [
      6,
      0
    ],
    [
      4,
      2
    ]
  ],
  "rows": 12
}


## Local Aer sampling

Use 128 shots per pair for this small demonstration; the full study uses 1,024. Keep the ideal-trained classifier and threshold. Finite-shot sampling here is not a physical-device noise model.

In [5]:
with tempfile.TemporaryDirectory(dir=ARTIFACTS, prefix="notebook-aer-") as temporary:
    measured = quantum.measured_kernel(
        c["reduced"] * angle_max,
        a["reduced"] * angle_max,
        "z",
        shots=128,
        seed=42,
        batch_size=64,
        directory=Path(temporary),
    )
measured_scores = features.scores(classifier, measured)
measured_predictions = core.predict(measured_scores, threshold)
print("Synthetic measured-quantum metrics:")
print(json.dumps(core.metrics(y_test, measured_predictions, measured_scores), indent=2))
print("Mean absolute kernel error:", float(np.abs(measured - ideal_kernel).mean()))
print("Prediction changes:", int((measured_predictions != ideal_predictions).sum()))

Synthetic measured-quantum metrics:
{
  "accuracy": 0.8333333333333334,
  "balanced_accuracy": 0.8333333333333333,
  "precision": 1.0,
  "recall": 0.6666666666666666,
  "false_positive_rate": 0.0,
  "f1": 0.8,
  "average_precision": 1.0,
  "confusion_matrix": [
    [
      6,
      0
    ],
    [
      2,
      4
    ]
  ],
  "rows": 12
}
Mean absolute kernel error: 0.031940412381934964
Prediction changes: 2


## Full study report, separate from the demonstration

Read the actual, versioned dataset experiment. Its evaluation sets are continuity checks because previous results have informed the research. Separate text benefit, reduction loss and kernel effect; do not claim a fresh independent test or quantum advantage.

In [6]:
study_path = ROOT / "reports/benchmark-v2-text/portfolio-text-20261028-verified.json"
study = json.loads(study_path.read_text(encoding="utf-8"))
assert study["status"] == "complete"
assert len(study["runs"]) == 42
print("Study:", study["run_id"], study["conclusion"])
for family, selected in study["selection"].items():
    print("Selected", family, selected)
    for partition in ("test_summary", "external_summary"):
        point = next(r for r in study[partition] if all(r[k] == v for k, v in selected.items()))
        print(
            partition,
            {
                m: round(point["metrics"][m]["mean"], 4)
                for m in ("accuracy", "recall", "false_positive_rate", "average_precision")
            },
        )
for partition, comparisons in study["comparisons"].items():
    print(partition, {name: value["ap_assessment"] for name, value in comparisons.items()})

Study: portfolio-text-20261028-verified target_not_met
Selected classical {'model': 'full_logistic', 'train_size': 100}
test_summary {'accuracy': 0.7852, 'recall': 0.5511, 'false_positive_rate': 0.0986, 'average_precision': 0.7268}
external_summary {'accuracy': 0.6524, 'recall': 0.4735, 'false_positive_rate': 0.1943, 'average_precision': 0.686}
Selected quantum {'model': 'quantum', 'train_size': 100}
test_summary {'accuracy': 0.7772, 'recall': 0.5235, 'false_positive_rate': 0.0969, 'average_precision': 0.7097}
external_summary {'accuracy': 0.6549, 'recall': 0.4649, 'false_positive_rate': 0.1825, 'average_precision': 0.6769}
test {'text_benefit': 'positive_difference', 'reduction_effect': 'negative_difference', 'kernel_effect': 'positive_difference', 'selected_classical': 'positive_difference'}
external {'text_benefit': 'positive_difference', 'reduction_effect': 'negative_difference', 'kernel_effect': 'positive_difference', 'selected_classical': 'positive_difference'}


## Interpretation

Read paired confidence intervals alongside training-seed dispersion. Measurement comparisons condition on one shot realization per fitted seed; they do not estimate all shot-noise uncertainty. SVD can discard useful class information despite retaining substantial variance.

The historical hardware experiment remains separate. This phase makes no production-readiness or hackathon-submission claim. See [the protocol and reproduction guide](../docs/benchmark-v2-text.md).